In [1]:
import pandas as pd 

gen_data = pd.read_csv('/home/s6moakba/gen_train_res_14_cv_MPW.csv')


In [2]:
gen_data

,blank,original_terms,original_text,prompt,aspectTerms_old,generated_text
0,"Best of all is the warm [MASK] , the [MASK] is...","['vibe', 'owner', 'service']","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'vibe', 'polarity': 'positive'}, {'t...","[""ambiance"", ""host"", ""service""]"
1,"Best of all is the warm [MASK] , the [MASK] is...","['vibe', 'owner', 'service']","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'vibe', 'polarity': 'positive'}, {'t...","[""ambiance"", ""host"", ""delivery""]"
2,"Best of all is the warm [MASK] , the [MASK] is...","['vibe', 'owner', 'service']","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'vibe', 'polarity': 'positive'}, {'t...","[""atmosphere"", ""staff"", ""service""]"
3,"Unfortunately , the [MASK] is outstanding , bu...",['food'],"Unfortunately, the food is outstanding, but ev...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'food', 'polarity': 'positive'}]","[""food""]"
4,"Unfortunately , the [MASK] is outstanding , bu...",['food'],"Unfortunately, the food is outstanding, but ev...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'food', 'polarity': 'positive'}]","[""food""]"
...,...,...,...,...,...,...
3067,What makes this restaurant special are the aut...,"['sichuan cooking', 'chongqing hotpot']",What makes this restaurant special are the aut...,\nYour task is to replace the [MASK] in the fo...,"[{'term': 'sichuan cooking', 'polarity': 'posi...","[""cuisine"", ""dishes"", ""Japanese"", ""sushi""]"
3068,What makes this restaurant special are the aut...,"['sichuan cooking', 'chongqing hotpot']",What makes this restaurant special are the aut...,\nYour task is to replace the [MASK] in the fo...,"[{'term': 'sichuan cooking', 'polarity': 'posi...","[""Japanese"", ""dishes"", ""Japanese"", ""cuisine""]"
3069,A wonderful [MASK] !,['place'],A wonderful place!,\nYour task is to replace the [MASK] in the fo...,"[{'term': 'place', 'polarity': 'positive'}]","[""experience""]"
3070,A wonderful [MASK] !,['place'],A wonderful place!,\nYour task is to replace the [MASK] in the fo...,"[{'term': 'place', 'polarity': 'positive'}]","[""meal""]"


In [3]:
import ast

def safe_literal_eval(val):
	try:
		return ast.literal_eval(val)
	except (ValueError, SyntaxError):
		return val

gen_data['original_terms'] = gen_data['original_terms'].apply(safe_literal_eval)
gen_data['generated_text'] = gen_data['generated_text'].apply(safe_literal_eval)
gen_data['aspectTerms_old'] = gen_data['aspectTerms_old'].apply(safe_literal_eval)

In [4]:
row = gen_data.iloc[16]
print(row['original_terms'])
print(row['aspectTerms_old'])
print(row['blank'])
print(row['generated_text'])

['food', 'served']
[{'term': 'food', 'polarity': 'positive'}, {'term': 'served', 'polarity': 'positive'}]
The [MASK] always tastes fresh and [MASK] promptly .
['coffee', 'serves']


In [5]:
import re
def collapse_consecutive_masks(text: str) -> str:
    """
    Replace consecutive [MASK] tokens (with optional spaces) by a single [MASK].
    Example:
        "The [MASK] [MASK] is nice" -> "The [MASK] is nice"
        "The [MASK]    [MASK]   [MASK]" -> "The [MASK]"
    """
    pattern = r'\[MASK\](?:\s*\[MASK\])+'  # [MASK] followed by one or more [MASK] with optional spaces
    return re.sub(pattern, '[MASK]', text)
gen_data["blank"] = gen_data["blank"].apply(collapse_consecutive_masks)


In [6]:
def group_generated_terms(original_terms, generated_text):
    """
    For each original term in `original_terms`, split by whitespace and
    grab that many items from `generated_text`. If an item is a tuple,
    replace it with None. If any item in that group is None, the entire
    grouped term becomes None.
    """
    grouped = []
    idx = 0
    for term in original_terms:
        n_words = len(term.split())  # how many words in this original term
        chunk = []
        
        # Extract exactly n_words from generated_text
        for gt in generated_text[idx : idx + n_words]:
            if isinstance(gt, tuple):
                chunk.append(None)
            else:
                chunk.append(gt)
        
        idx += n_words
        
        # If there's any None in the chunk, set the entire grouped term to None
        if any(c is None for c in chunk):
            grouped.append(None)
        else:
            grouped.append(" ".join(chunk))
    
    return grouped

# Apply the function for each row in df
gen_data["generated_text"] = gen_data.apply(lambda row: group_generated_terms(row["original_terms"], row["generated_text"]),axis=1)


In [7]:
gen_data = gen_data[~gen_data["generated_text"].apply(lambda x: any(elem is None for elem in x))]
gen_data.shape

(3072, 6)

In [8]:
original_data = pd.read_csv('/home/s6moakba/InstructABSA/Dataset/SemEval14/Train/Restaurants_Train.csv')
original_data.head()

,sentenceId,raw_text,aspectTerms,aspectCategories
0,3121,But the staff was so horrible to us.,"[{'term': 'staff', 'polarity': 'negative'}]","[{'category': 'service', 'polarity': 'negative'}]"
1,2777,"To be completely fair, the only redeeming fact...","[{'term': 'food', 'polarity': 'positive'}]","[{'category': 'food', 'polarity': 'positive'},..."
2,1634,"The food is uniformly exceptional, with a very...","[{'term': 'food', 'polarity': 'positive'}, {'t...","[{'category': 'food', 'polarity': 'positive'}]"
3,2534,Where Gabriela personaly greets you and recomm...,"[{'term': 'noaspectterm', 'polarity': 'none'}]","[{'category': 'service', 'polarity': 'positive'}]"
4,583,"For those that go once and don't enjoy it, all...","[{'term': 'noaspectterm', 'polarity': 'none'}]","[{'category': 'anecdotes/miscellaneous', 'pola..."


In [9]:
# Find duplicated generated_text
duplicated_texts = gen_data['generated_text'].duplicated(keep='first')

# Filter out rows with duplicated generated_text
gen_data = gen_data[~duplicated_texts]

gen_data.shape

(2041, 6)

In [10]:
filtered_data = gen_data[gen_data['original_terms'].apply(len) != gen_data['generated_text'].apply(len)]
gen_data.drop( filtered_data.index , inplace=True)

In [11]:
roww = gen_data.iloc[14]
print(roww['original_terms'])
print(roww['generated_text'])
print(roww['blank'])

['food', 'served']
['waiter', 'serves']
The [MASK] always tastes fresh and [MASK] promptly .


In [12]:
def replace_terms(row):
    aspect_terms = row['aspectTerms_old']
    generated_text = row['generated_text']
    for i, term in enumerate(aspect_terms):
        term['term'] = generated_text[i]
    return aspect_terms

gen_data['aspectTerms'] = gen_data.apply(replace_terms, axis=1)

In [13]:
def replace_mask(row):
    text = row['blank']
    for term in row['generated_text']:
        text = text.replace('[MASK]', term, 1)
    return text

gen_data['blank'] = gen_data.apply(replace_mask, axis=1)
gen_data.head()

,blank,original_terms,original_text,prompt,aspectTerms_old,generated_text,aspectTerms
0,"Best of all is the warm ambiance , the host is...","[vibe, owner, service]","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'ambiance', 'polarity': 'positive'},...","[ambiance, host, service]","[{'term': 'ambiance', 'polarity': 'positive'},..."
1,"Best of all is the warm ambiance , the host is...","[vibe, owner, service]","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'ambiance', 'polarity': 'positive'},...","[ambiance, host, delivery]","[{'term': 'ambiance', 'polarity': 'positive'},..."
2,"Best of all is the warm atmosphere , the staff...","[vibe, owner, service]","Best of all is the warm vibe, the owner is sup...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'atmosphere', 'polarity': 'positive'...","[atmosphere, staff, service]","[{'term': 'atmosphere', 'polarity': 'positive'..."
3,"Unfortunately , the food is outstanding , but ...",[food],"Unfortunately, the food is outstanding, but ev...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'food', 'polarity': 'positive'}]",[food],"[{'term': 'food', 'polarity': 'positive'}]"
5,"Unfortunately , the ambiance is outstanding , ...",[food],"Unfortunately, the food is outstanding, but ev...",\nYour task is to replace the [MASK] in the fo...,"[{'term': 'ambiance', 'polarity': 'positive'}]",[ambiance],"[{'term': 'ambiance', 'polarity': 'positive'}]"


In [14]:
roww = gen_data.iloc[14]
print(roww['original_terms'])
print(roww['generated_text'])
print(roww['blank'])

['food', 'served']
['waiter', 'serves']
The waiter always tastes fresh and serves promptly .


In [62]:
gen_data.shape

(2041, 7)

In [63]:
gen_data.drop(columns=['original_terms', 'aspectTerms_old','prompt'], inplace=True)

In [64]:
gen_data.rename(columns={'blank':'raw_text'},inplace=True)

In [65]:
gen_data.head()

,raw_text,original_text,generated_text,aspectTerms
0,"Best of all is the warm ambiance , the host is...","Best of all is the warm vibe, the owner is sup...","[ambiance, host, service]","[{'term': 'ambiance', 'polarity': 'positive'},..."
1,"Best of all is the warm ambiance , the host is...","Best of all is the warm vibe, the owner is sup...","[ambiance, host, delivery]","[{'term': 'ambiance', 'polarity': 'positive'},..."
2,"Best of all is the warm atmosphere , the staff...","Best of all is the warm vibe, the owner is sup...","[atmosphere, staff, service]","[{'term': 'atmosphere', 'polarity': 'positive'..."
3,"Unfortunately , the food is outstanding , but ...","Unfortunately, the food is outstanding, but ev...",[food],"[{'term': 'food', 'polarity': 'positive'}]"
5,"Unfortunately , the ambiance is outstanding , ...","Unfortunately, the food is outstanding, but ev...",[ambiance],"[{'term': 'ambiance', 'polarity': 'positive'}]"


In [66]:
gen_data['aspectCategories'] = gen_data['aspectTerms'].apply(lambda x: [{'category': 'general', 'polarity': 'neutral'}])

In [67]:
gen_data['sentenceId'] = gen_data.index

In [68]:
gen_data.to_csv('gen_train_res_14_format_cv_MPW.csv',index=False)

# Check Results

In [2]:
import pandas as pd
gen_data = pd.read_csv('gen_train_res_14_format_cv_MPW_1.csv')
gen_data.shape

(841, 6)